# 3.2 能不能根據當前需求，改變讀取比例？


剛才九成與一成是人指定的。現在準備一個「目前要找什麼」的數字卡 `[1,0]`，以及三個候選位置的線索 `[1,0]`、`[0,1]`、`[-1,0]`。我們希望第一個位置匹配最高，第二個其次，第三個最低，讓讀取隨需求改變。

需求卡叫 **query**，縮寫 q；候選的匹配卡叫 **key**。像用需求對照書目，但這裡每張都是實際數字，而不是人寫的關鍵詞。兩格僅是手動例子，不宣稱第一格代表顏色。

匹配方法是對應格相乘再相加，叫**點積**：三個分數依序是 `1×1+0×0=1`、0、-1。再用上一章的 softmax 將分數變成非負、總和為 1 的讀取比例。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

q = torch.tensor([1.0, 0.0])
keys = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
scores = q @ keys.T
weights = scores.softmax(dim=0)
print(scores)
print(weights)

tensor([ 1.,  0., -1.])
tensor([0.6652, 0.2447, 0.0900])


`q @ keys.T` 得 `[1,0,-1]`；交換 key 的列欄，讓一次乘法算三個匹配。比例約 `[0.665,0.245,0.090]`。第一個最多，另外兩個仍有份。

若需求改 `[0,1]`，分數變 `[0,1,0]`，第二個位置取得最多。候選位置沒有改，偏好已因需求改變；這就是用內容決定讀法。

點積也受數字大小影響。將第一個 key 改 `[2,0]`，方向沒變，對 `[1,0]` 的分數卻變 2。它不是只量方向的相似度，稍後還要控制分數尺度。

練習先改 q 成 `[0,1]`，核對第二個最高；再只把第三個 key 改成 `[-1,1]`，後兩者都得 1，各分到約 0.422。需求和候選線索都能改比例。

<details>
<summary>補充與重做</summary>

本節只算匹配與比例，沒有取回內容，也先不限制讀哪些位置。分數尚未除以特徵數平方根；[3.5](https://birdhackor.github.io/tiny-perceptron-vlm/3.5.html) 再加入尺度規則，[3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 加入防偷看許可。softmax 見 [1.7](https://birdhackor.github.io/tiny-perceptron-vlm/1.7.html)。

</details>
